# Sentinel-2 Data Access with Google Earth Engine (GEE)

This notebook uses Google Earth Engine (GEE) to access and process Sentinel-2 imagery in the cloud for wildfire monitoring, rather than direct local downloads.

## Objectives:
- Authenticate and initialize Google Earth Engine
- Define the area of interest (AOI)
- Query Sentinel-2 imagery using date and cloud-cover filters
- Visualize Sentinel-2 composites over the AOI
- Organize and catalog image metadata 

## 1. Initialize Environment and Import Libraries

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# GEE and geospatial libraries
import ee
import geemap
import folium

# Authenticate and initialize GEE
#ee.Authenticate()    # This will open a browser for you to log in (run once per session)
ee.Initialize(project='your-project-id')  

# Add src directory to path
sys.path.insert(0, '../src')
from sentinel_data import SentinelGEEData
from utils import create_directories, save_config

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## 2. Set Up Project Directories and Configuration

In [6]:
# Define project paths
PROJECT_ROOT = Path('../').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DATA_DIR = DATA_DIR / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'

print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Directory: {DATA_DIR}")

# Create directories
create_directories(str(PROJECT_ROOT))
print("✓ Project directories created/verified")

Project Root: C:\Users\htamiminia\monitoring
Data Directory: C:\Users\htamiminia\monitoring\data
✓ Project directories created/verified


## 3. Configure Area of Interest and Time Period

In [7]:
# Define area of interest (AOI) as bounding box
# Format: [min_lon, min_lat, max_lon, max_lat]
AOI_BBOX = [-123.075, 41.689, -122.685, 41.927]  

# Convert bbox to GEE geometry
min_lon, min_lat, max_lon, max_lat = AOI_BBOX
AOI_GEE = ee.Geometry.BBox(min_lon, min_lat, max_lon, max_lat)

# Time period configuration
START_DATE = '2018-01-01'
END_DATE = '2025-12-31'
MAX_CLOUD_COVER = 20  # Percentage

print(f"Area of Interest: {AOI_BBOX}")
print(f"Date Range: {START_DATE} to {END_DATE}")
print(f"Max Cloud Cover: {MAX_CLOUD_COVER}%")

# Save configuration
config = {
    'aoi_bbox': AOI_BBOX,
    'start_date': START_DATE,
    'end_date': END_DATE,
    'max_cloud_cover': MAX_CLOUD_COVER,
    'data_source': 'Google Earth Engine (GEE)',
    'created_at': datetime.now().isoformat()
}
save_config(config, '../config.json')
print("✓ Configuration saved")

Area of Interest: [-123.075, 41.689, -122.685, 41.927]
Date Range: 2018-01-01 to 2025-12-31
Max Cloud Cover: 20%
✓ Configuration saved


## 4. Initialize GEE and Sentinel-2 Access

In [ ]:
# Authenticate with Google Earth Engine
print("Initializing Google Earth Engine...")
try:
    # Authenticate
    ee.Authenticate()
    print("✓ Authentication successful")
except Exception as e:
    print(f"Note: GEE may already be authenticated: {e}")

# Initialize with your GEE project (replace with your project name if available)
try:
    ee.Initialize(project='your-project-id')  # Replace with your GEE project
    print("✓ GEE Initialized with project")
except:
    ee.Initialize()
    print("✓ GEE Initialized with default settings")

# Initialize Sentinel-2 GEE accessor
sentinel_gee = SentinelGEEData()
print("✓ Sentinel-2 GEE Data accessor initialized")

Initializing Google Earth Engine...
✓ Authentication successful
✓ GEE Initialized with project
✓ Sentinel-2 GEE Data accessor initialized


## 5. Load and Visualize Sentinel-2 Imagery from GEE

In [ ]:
# Load Sentinel-2 imagery from GEE
# ee.Initialize(project='your-project-id')  
print("Loading Sentinel-2 imagery from Google Earth Engine...")

s2_collection = sentinel_gee.load_sentinel2_collection(
    aoi=AOI_GEE,
    start_date=START_DATE,
    end_date=END_DATE,
    cloud_cover=MAX_CLOUD_COVER
)

# Get collection info
n_images = s2_collection.size().getInfo()
print(f"✓ Found {n_images} Sentinel-2 images")

# Create geemap visualization
Map = geemap.Map(center=[(AOI_BBOX[1] + AOI_BBOX[3])/2, 
                          (AOI_BBOX[0] + AOI_BBOX[2])/2], zoom=10)

# Create RGB composite (using median)
rgb_composite = s2_collection.select(['B4', 'B3', 'B2']).median()

# Add layer to map
vis_params = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 0.3,
    'gamma': 1.4
}

Map.addLayer(rgb_composite.clip(AOI_GEE), vis_params, 'RGB Composite')
Map.addLayer(AOI_GEE, {'color': 'FF0000'}, 'Area of Interest')

Map


Loading Sentinel-2 imagery from Google Earth Engine...
✓ Found 581 Sentinel-2 images


Map(center=[41.808, -122.88], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprig…

## 6. Catalog Retrieved Sentinel-2 Metadata

In [11]:
print("Extracting Sentinel-2 collection metadata...")

n_images = s2_collection.size().getInfo()
print(f"Number of images in s2_collection: {n_images}")

if n_images > 0:
    image_ids = s2_collection.aggregate_array('system:index').getInfo()
    image_dates = s2_collection.aggregate_array('system:time_start').getInfo()
    cloud_percentages = s2_collection.aggregate_array('CLOUDY_PIXEL_PERCENTAGE').getInfo()

    print("Sample image_ids:", image_ids[:3])
    print("Sample image_dates:", image_dates[:3])
    print("Sample cloud_percentages:", cloud_percentages[:3])

    from datetime import datetime as dt
    dates_readable = [dt.fromtimestamp(t/1000).strftime('%Y-%m-%d') for t in image_dates]

    catalog_data = []
    for i, (img_id, date, cloud) in enumerate(zip(image_ids, dates_readable, cloud_percentages)):
        catalog_data.append({
            'index': i + 1,
            'date': date,
            'image_id': img_id,
            'cloud_cover_%': round(cloud, 2)
        })

    catalog_df = pd.DataFrame(catalog_data)
    print("\nSentinel-2 Collection Catalog:")
    print(catalog_df.to_string(index=False))
    print(f"\nTotal images: {len(catalog_df)}")
    if not catalog_df.empty and 'date' in catalog_df.columns:
        print(f"Date range: {catalog_df['date'].min()} to {catalog_df['date'].max()}")
    else:
        print("No images found or 'date' column missing.")
else:
    print("No images found in the collection. Check your AOI, date range, and filters.")

Extracting Sentinel-2 collection metadata...
Number of images in s2_collection: 581
Sample image_ids: ['20180113T190729_20180113T190732_T10TEM', '20180202T190609_20180202T191214_T10TDM', '20180202T190609_20180202T191214_T10TEM']
Sample image_dates: [1515870765414, 1517598770291, 1517598766566]
Sample cloud_percentages: [14.918369, 0.579331, 0.739832]

Sentinel-2 Collection Catalog:
 index       date                               image_id  cloud_cover_%
     1 2018-01-13 20180113T190729_20180113T190732_T10TEM          14.92
     2 2018-02-02 20180202T190609_20180202T191214_T10TDM           0.58
     3 2018-02-02 20180202T190609_20180202T191214_T10TEM           0.74
     4 2018-02-07 20180207T190541_20180207T190540_T10TDM          14.00
     5 2018-02-07 20180207T190541_20180207T190540_T10TEM          18.78
     6 2018-02-12 20180212T190509_20180212T190505_T10TEM          16.56
     7 2018-03-19 20180319T190101_20180319T190740_T10TDM           5.06
     8 2018-03-19 20180319T190101_20180

## Summary
This notebook sets up and validates Sentinel-2 data access through GEE for the project AOI.
It defines the spatial and temporal configuration, initializes GEE access, loads filtered Sentinel-2 imagery, and catalogs collection metadata for downstream processing.

The workflow focuses on:
- Initializing environment and GEE authentication
- Defining AOI, date range, and cloud-cover threshold
- Querying Sentinel-2 imagery from GEE
- Visualizing an RGB composite for quick quality inspection
- Exporting configuration and metadata needed for the pipeline

## Sentinel-2 Data Source
- Dataset: `COPERNICUS/S2_SR_HARMONIZED`
- Product: Surface reflectance (atmospherically corrected)
- Spatial resolution: 10 m
- Bands used: B4 (Red), B8 (NIR) for Normalized Difference Vegetation Index (NDVI) computation

## Next Steps

1. Run **02_ndvi_calculation.ipynb** to compute NDVI from GEE data
2. Prepare time series data for Long Short-Term Memory (LSTM) training
3. Train LSTM-based model for wildfire anomaly detection and evaluate performance
